# EA Sports FC 26 — Análisis de correlaciones

**Introducción a la Ciencia de Datos — Equipo Videojuegos**

**Etapa 2 del proyecto.** En el notebook `01_carga_limpieza_orden.ipynb` dejamos los
datos limpios y ordenados. Ahora usamos esos datos para responder la pregunta del
proyecto:

> **¿Qué atributos de la carta determinan la valoración general (OVR)?**

Este notebook **no repite nada del anterior**: arranca leyendo los CSV ya limpios de
`data/processed/`. Si esos archivos no existen, hay que correr primero el notebook 01.

## 0. Preparación

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Buscamos hacia arriba la carpeta del proyecto (la que contiene data/processed)
DIR_BASE = Path.cwd()
for carpeta in [DIR_BASE, *DIR_BASE.parents]:
    if (carpeta / "data" / "processed").is_dir():
        DIR_BASE = carpeta
        break
else:
    raise FileNotFoundError("No encuentro data/processed. ¿Corriste antes el notebook 01?")

DIR_LIMPIO = DIR_BASE / "data" / "processed"
DIR_FIGURAS = DIR_BASE / "output" / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)


def leer_dataset_limpio(nombre_archivo: str) -> pd.DataFrame:
    """Lee un CSV de data/processed/ devolviendo las listas vacías como '' y no como NaN."""
    df = pd.read_csv(DIR_LIMPIO / nombre_archivo, encoding="utf-8")
    for col in ["alternative_positions", "play_styles"]:
        if col in df.columns:
            df[col] = df[col].fillna("")
    return df


campo = leer_dataset_limpio("jugadores_campo_limpio.csv")
arqueros = leer_dataset_limpio("arqueros_limpio.csv")

print(f"Jugadores de campo: {len(campo):,}".replace(",", "."))
print(f"Arqueros:           {len(arqueros):,}".replace(",", "."))

### Estilo de los gráficos

Usamos la misma paleta que en el notebook 01 para que la presentación se vea coherente.
Para las correlaciones agregamos una escala **divergente** (azul ↔ rojo con gris en el
medio), que es la correcta cuando los valores pueden ser negativos o positivos: el gris
marca el cero, el azul las relaciones positivas y el rojo las negativas.

In [ ]:
SUPERFICIE = "#fcfcfb"
TINTA = "#0b0b0b"
TINTA_SUAVE = "#52514e"
TINTA_TENUE = "#898781"
GRILLA = "#e1e0d9"
AZUL = "#2a78d6"
AZUL_CLARO = "#b7d3f6"
NARANJA = "#eb6834"
AQUA = "#1baf7a"
GRIS = "#c3c2b7"

# Escala divergente: rojo (negativo) ↔ gris (cero) ↔ azul (positivo)
ESCALA_CORR = LinearSegmentedColormap.from_list(
    "corr", ["#a32a29", "#e34948", "#f0efec", "#86b6ef", "#184f95"]
)

mpl.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
    "savefig.facecolor": SUPERFICIE, "font.family": "sans-serif", "font.size": 11,
    "text.color": TINTA, "axes.labelcolor": TINTA_SUAVE, "axes.edgecolor": GRIS,
    "xtick.color": TINTA_TENUE, "ytick.color": TINTA_TENUE,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRILLA, "grid.linewidth": 0.8,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})


def poner_titulo(ax, titulo: str, subtitulo: str, alto=(1.14, 1.045)) -> None:
    """Título en negrita + subtítulo gris arriba del gráfico, sin superponerse."""
    ax.text(0, alto[0], titulo, transform=ax.transAxes, fontsize=14,
            fontweight="bold", color=TINTA, va="bottom", ha="left")
    ax.text(0, alto[1], subtitulo, transform=ax.transAxes, fontsize=10.5,
            color=TINTA_SUAVE, va="bottom", ha="left")


def coma(valor: float, decimales: int = 2) -> str:
    """Formatea 0.896 como '0,90' (separador decimal al estilo rioplatense)."""
    return f"{valor:.{decimales}f}".replace(".", ",")


def guardar(fig, nombre: str) -> None:
    fig.savefig(DIR_FIGURAS / nombre)
    print(f"✓ {nombre}")

---
# PARTE 1 — ¿Qué es una correlación?

El **coeficiente de correlación** mide si dos variables se mueven juntas. Va de **-1 a
+1**:

| Valor | Qué significa |
|---|---|
| **+1** | Cuando una sube, la otra sube siempre, de forma perfecta |
| **+0,7 a +0,9** | Relación fuerte: casi siempre suben juntas |
| **+0,3 a +0,5** | Relación débil: a veces sí, a veces no |
| **0** | No tienen ninguna relación |
| **Negativo** | Cuando una sube, la otra baja |

Nosotros vamos a calcular la correlación **de cada atributo con el OVR**. El atributo
con el coeficiente más alto es el que mejor explica la valoración de una carta.

> ⚠️ **Advertencia importante:** correlación **no** es causalidad. Que dos cosas se
> muevan juntas no prueba que una cause la otra. Volvemos a esto en las conclusiones.

## 1.1 Definimos los grupos de atributos

In [ ]:
CARAS = ["pac", "sho", "pas", "dri", "def", "phy"]

ATRIBUTOS_DETALLE = [
    "acceleration", "sprint_speed", "positioning", "finishing", "shot_power",
    "long_shots", "volleys", "penalties", "vision", "crossing", "free_kick_accuracy",
    "short_passing", "long_passing", "curve", "dribbling", "agility", "balance",
    "reactions", "ball_control", "composure", "interceptions", "heading_accuracy",
    "def_awareness", "standing_tackle", "sliding_tackle", "jumping", "stamina",
    "strength", "aggression",
]

CARACTERISTICAS = ["age", "height_cm", "weight_kg", "weak_foot", "skill_moves",
                   "n_play_styles", "n_play_styles_plus"]

TODOS = CARAS + ATRIBUTOS_DETALLE + CARACTERISTICAS

# Nombres en castellano, para que los gráficos se entiendan en la presentación
NOMBRES = {
    "ovr": "OVR", "pac": "PAC (ritmo)", "sho": "SHO (tiro)", "pas": "PAS (pase)",
    "dri": "DRI (regate)", "def": "DEF (defensa)", "phy": "PHY (físico)",
    "reactions": "Reacciones", "composure": "Serenidad", "short_passing": "Pase corto",
    "ball_control": "Control del balón", "long_passing": "Pase largo",
    "dribbling": "Regate (detalle)", "vision": "Visión de juego", "shot_power": "Potencia de tiro",
    "crossing": "Centros", "jumping": "Salto", "curve": "Comba", "agility": "Agilidad",
    "finishing": "Definición", "positioning": "Posicionamiento", "long_shots": "Tiros lejanos",
    "volleys": "Voleas", "penalties": "Penales", "free_kick_accuracy": "Tiros libres",
    "acceleration": "Aceleración", "sprint_speed": "Velocidad punta", "balance": "Equilibrio",
    "interceptions": "Intercepciones", "heading_accuracy": "Cabeceo",
    "def_awareness": "Corte defensivo", "standing_tackle": "Entrada", "sliding_tackle": "Barrida",
    "stamina": "Resistencia", "strength": "Fuerza", "aggression": "Agresividad",
    "age": "Edad", "height_cm": "Altura", "weight_kg": "Peso", "weak_foot": "Pie malo",
    "skill_moves": "Filigranas", "n_play_styles": "Cantidad de PlayStyles",
    "n_play_styles_plus": "PlayStyles+", "gk_diving": "DIV (estiradas)",
    "gk_handling": "HAN (blocaje)", "gk_kicking": "KIC (saque)",
    "gk_reflexes": "REF (reflejos)", "gk_speed": "SPD (velocidad)",
    "gk_positioning": "POS (colocación)",
}

# Las 6 caras son las únicas que el jugador ve en la carta dentro del juego
EN_LA_CARTA = set(CARAS)

print(f"{len(TODOS)} variables a correlacionar con el OVR.")

## 1.2 La tabla de correlaciones

Esta es la tabla que responde la pregunta del proyecto. La calculamos **sólo sobre los
jugadores de campo**: como demostramos en el notebook 01, mezclar arqueros arruinaría
el resultado.

In [ ]:
correlaciones = campo[TODOS].corrwith(campo["ovr"]).sort_values(ascending=False)

tabla = pd.DataFrame({
    "atributo": [NOMBRES.get(a, a) for a in correlaciones.index],
    "coeficiente": correlaciones.values.round(3),
    "en la carta": ["Sí" if a in EN_LA_CARTA else "No" for a in correlaciones.index],
})
print("TOP 15 — atributos que más acompañan al OVR:\n")
print(tabla.head(15).to_string(index=False))
print("\nLos 5 que menos:\n")
print(tabla.tail(5).to_string(index=False))

**Primer hallazgo.** El atributo con la correlación más alta es **Reacciones (0,896)**,
y no aparece en la cara de la carta. Le siguen Pase corto, Serenidad y Control del
balón, que tampoco aparecen. La primera cara visible es PAS, recién en quinto lugar.

Y mirá el fondo de la tabla: **PAC (ritmo) apenas llega a 0,24**, aunque sea el número
más grande de la carta. La altura tiene correlación **negativa** (-0,08): ser alto no
mejora la valoración.

## 1.3 Validación: ¿el resultado es confiable?

El coeficiente que usamos (Pearson) supone que la relación entre las variables es una
línea recta. Si la relación fuera curva, daría un número engañoso. Para descartarlo lo
comparamos con **Spearman**, que no supone nada sobre la forma: si los dos coinciden,
la relación es lineal y el resultado es sólido.

In [ ]:
comparacion = pd.DataFrame({
    "Pearson": campo[TODOS].corrwith(campo["ovr"]),
    "Spearman": campo[TODOS].corrwith(campo["ovr"], method="spearman"),
})
comparacion["diferencia"] = (comparacion["Pearson"] - comparacion["Spearman"]).abs()
comparacion = comparacion.sort_values("Pearson", ascending=False)

dif_top = comparacion.head(15)["diferencia"].max()
dif_total = comparacion["diferencia"].max()

print("Los 8 más correlacionados:\n")
print(comparacion.head(8).round(3).to_string())
print(f"\nDiferencia máxima dentro del top 15: {dif_top:.3f}")
print("✓ Los dos métodos coinciden en los atributos que más importan: el resultado es confiable.\n")

print(f"Diferencia máxima en todo el conjunto: {dif_total:.3f}")
print("Las mayores diferencias están en los atributos defensivos:\n")
print(comparacion.sort_values("diferencia", ascending=False).head(4).round(3).to_string())

**Resultado.** En los atributos que más importan los dos métodos coinciden (diferencia
máxima 0,046), así que el ranking es sólido.

Donde sí difieren es en los **atributos defensivos** (Entrada 0,33 vs 0,43;
Intercepciones 0,35 vs 0,45). No es un error: es una pista. Esos atributos no tienen
una relación de línea recta con el OVR porque **dependen de la posición** — un delantero
de 90 de media puede tener 20 de entrada sin que eso le baje la valoración. Lo
confirmamos más adelante, en el gráfico 10.

---
# PARTE 2 — Los gráficos

## Gráfico 7 — Matriz de correlación

El **diagrama de correlación** clásico. Cada celda cruza dos variables y muestra su
coeficiente. La diagonal siempre vale 1 (una variable consigo misma).

Se lee de dos formas: la **primera fila** dice cuánto acompaña cada atributo al OVR, y
**el resto del cuadro** muestra cuánto se parecen los atributos entre sí.

In [ ]:
SELECCION = ["ovr"] + CARAS + ["reactions", "short_passing", "composure", "ball_control"]
matriz = campo[SELECCION].corr()
etiquetas = [NOMBRES[a] for a in SELECCION]

fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(matriz.values, cmap=ESCALA_CORR, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))

for i in range(len(SELECCION)):
    for j in range(len(SELECCION)):
        valor = matriz.values[i, j]
        # Texto blanco sobre las celdas oscuras, negro sobre las claras
        color = "white" if abs(valor) > 0.62 else TINTA
        peso = "bold" if (i == 0 or j == 0) and i != j else "normal"
        ax.text(j, i, coma(valor), ha="center", va="center",
                fontsize=9.5, color=color, fontweight=peso)

ax.set_xticks(range(len(SELECCION)), etiquetas, rotation=45, ha="right",
              fontsize=10, color=TINTA_SUAVE)
ax.set_yticks(range(len(SELECCION)), etiquetas, fontsize=10, color=TINTA_SUAVE)
ax.set_xticks(np.arange(len(SELECCION) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(SELECCION) + 1) - 0.5, minor=True)
ax.grid(which="minor", color=SUPERFICIE, linewidth=2)
ax.tick_params(which="minor", length=0)
for lado in ax.spines.values():
    lado.set_visible(False)

barra = fig.colorbar(im, ax=ax, shrink=0.62, pad=0.03, ticks=[-1, -0.5, 0, 0.5, 1])
barra.ax.set_yticklabels(["-1", "-0,5", "0", "0,5", "1"], fontsize=9, color=TINTA_TENUE)
barra.outline.set_visible(False)
barra.ax.set_title("coef.", fontsize=9, color=TINTA_TENUE, pad=8)

poner_titulo(ax, "Matriz de correlación con el OVR",
             "Primera fila: cuánto acompaña cada atributo al OVR. El resto: cuánto se parecen entre sí.",
             alto=(1.16, 1.08))
ax.text(0, -0.27,
        "Las cuatro últimas columnas son atributos de detalle que NO aparecen en la cara de la carta.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "07_matriz_correlacion.png")
plt.show()

**Qué se ve.** En la primera fila, Reacciones (0,90) le gana a todas las caras visibles.
Y en el resto del cuadro aparece algo importante: **PAS y DRI están correlacionados
entre sí en 0,86**. No son informaciones independientes, y eso hay que tenerlo en
cuenta antes de sacar conclusiones sobre el efecto de cada una por separado.

## Gráfico 8 — Ranking: qué atributo manda

La matriz es buena para ver el conjunto, pero para responder *"¿cuál es el que más
importa?"* conviene un ranking ordenado. Pintamos de color los atributos que **no**
aparecen en la carta, para que se vea dónde quedan.

In [ ]:
# Mostramos los 10 más correlacionados + las 6 caras de la carta + altura y peso.
# Así se garantiza que las seis caras estén siempre en el gráfico: sin ellas el
# subtítulo hablaría de atributos que no se ven.
seleccion = list(dict.fromkeys(
    list(correlaciones.head(10).index) + CARAS + ["height_cm", "weight_kg"]
))
mostrar = correlaciones[seleccion].sort_values(ascending=False)

colores = [NARANJA if a in EN_LA_CARTA else AZUL for a in mostrar.index]

fig, ax = plt.subplots(figsize=(10, 8.2))
posiciones = list(range(len(mostrar)))[::-1]
ax.barh(posiciones, mostrar.values, color=colores, height=0.68)

for y, (nombre, valor) in zip(posiciones, mostrar.items()):
    color = NARANJA if nombre in EN_LA_CARTA else AZUL
    ax.text(valor + 0.015 if valor > 0 else valor - 0.015, y, coma(valor),
            ha="left" if valor > 0 else "right", va="center",
            fontsize=10.5, fontweight="bold", color=color)

ax.set_yticks(posiciones, [NOMBRES.get(a, a) for a in mostrar.index],
              fontsize=10.5, color=TINTA_SUAVE)
ax.axvline(0, color=GRIS, linewidth=1.2)
ax.set_xlim(-0.25, 1.02)
ax.set_xticks([-0.25, 0, 0.25, 0.5, 0.75, 1.0], ["-0,25", "0", "0,25", "0,50", "0,75", "1"])
ax.set_xlabel("Coeficiente de correlación con el OVR", labelpad=8)
ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=AZUL, label="No aparece en la carta"),
                   Patch(facecolor=NARANJA, label="Sí aparece en la carta")],
          loc="lower right", frameon=False, fontsize=10.5, labelcolor=TINTA_SUAVE,
          handlelength=1.1, handleheight=1.1)

poner_titulo(ax, "El atributo que más define el OVR no está en la carta",
             "Los diez más correlacionados, más las seis caras visibles de la carta y el físico, ordenados de mayor a menor.",
             alto=(1.09, 1.03))
ax.text(0, -0.105,
        "Los cuatro primeros no aparecen en la carta. PAC (0,24), que es el número más grande que ve el jugador, "
        "queda entre los últimos;\nla altura incluso da negativa (-0,08): ser alto no mejora la valoración.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "08_ranking_correlacion_ovr.png")
plt.show()

## Gráfico 9 — Qué significan esos números

Un coeficiente es abstracto. Para entenderlo hay que **ver** los datos. Acá dibujamos
cada uno de los 15.859 jugadores como un punto: a la izquierda el atributo más
correlacionado, a la derecha el menos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5), sharey=True)

paneles = [
    (axes[0], "reactions", AZUL, "El más correlacionado"),
    (axes[1], "pac", NARANJA, "El menos correlacionado"),
]

for ax, atributo, color, rotulo in paneles:
    r = campo[atributo].corr(campo["ovr"])
    ax.scatter(campo[atributo], campo["ovr"], s=5, alpha=0.12,
               color=color, edgecolors="none", zorder=2)

    # Recta de tendencia: resume hacia dónde va la nube de puntos
    pendiente, ordenada = np.polyfit(campo[atributo], campo["ovr"], 1)
    x = np.array([campo[atributo].min(), campo[atributo].max()])
    ax.plot(x, pendiente * x + ordenada, color=TINTA, linewidth=2, zorder=3)

    ax.text(0.04, 0.95, f"r = {coma(r)}", transform=ax.transAxes,
            fontsize=22, fontweight="bold", color=color, va="top", ha="left")
    ax.text(0.04, 0.845, rotulo, transform=ax.transAxes,
            fontsize=10.5, color=TINTA_SUAVE, va="top", ha="left")
    ax.set_xlabel(NOMBRES[atributo], labelpad=8)
    ax.set_xlim(10, 100)
    ax.grid(True, zorder=0)
    ax.set_axisbelow(True)

axes[0].set_ylabel("OVR (valoración general)")
axes[0].set_ylim(42, 96)

poner_titulo(axes[0], "Qué aspecto tiene una correlación fuerte y una débil",
             "Cada punto es uno de los 15.859 jugadores de campo. La línea negra resume la tendencia.",
             alto=(1.14, 1.05))
axes[0].text(0, -0.22,
             "Izquierda: los puntos forman una franja angosta y clara, la relación es fuerte. "
             "Derecha: es una nube dispersa, saber el ritmo\nde un jugador casi no dice nada sobre su OVR.",
             transform=axes[0].transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "09_dispersion_fuerte_vs_debil.png")
plt.show()

## Gráfico 10 — ¿La respuesta cambia según la posición?

Hasta acá miramos a todos los jugadores de campo juntos. Pero un delantero y un central
no se evalúan con la misma vara. Repetimos el cálculo **posición por posición**.

In [ ]:
POSICIONES = ["ST", "RW", "CAM", "CM", "CDM", "LM", "RB", "LB", "CB"]
ATRIBUTOS_POS = ["reactions", "composure", "sho", "pas", "dri", "def", "phy", "pac"]

matriz_pos = pd.DataFrame(
    {pos: {a: campo.loc[campo.position == pos, a].corr(campo.loc[campo.position == pos, "ovr"])
           for a in ATRIBUTOS_POS}
     for pos in POSICIONES}
)

fig, ax = plt.subplots(figsize=(10.5, 6))
im = ax.imshow(matriz_pos.values, cmap=ESCALA_CORR,
               norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1), aspect="auto")

for i in range(len(ATRIBUTOS_POS)):
    for j in range(len(POSICIONES)):
        valor = matriz_pos.values[i, j]
        color = "white" if abs(valor) > 0.62 else TINTA
        ax.text(j, i, coma(valor), ha="center", va="center", fontsize=10, color=color)

ax.set_xticks(range(len(POSICIONES)), POSICIONES, fontsize=11.5, color=TINTA_SUAVE)
ax.set_yticks(range(len(ATRIBUTOS_POS)), [NOMBRES[a] for a in ATRIBUTOS_POS],
              fontsize=10.5, color=TINTA_SUAVE)
ax.set_xticks(np.arange(len(POSICIONES) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(ATRIBUTOS_POS) + 1) - 0.5, minor=True)
ax.grid(which="minor", color=SUPERFICIE, linewidth=2)
ax.tick_params(which="minor", length=0)
for lado in ax.spines.values():
    lado.set_visible(False)

# Recuadro sobre la fila de Reacciones: es la única que no cambia
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-0.5, -0.5), len(POSICIONES), 1, fill=False,
                       edgecolor=AQUA, linewidth=2.5, zorder=5))
ax.text(len(POSICIONES) - 0.4, 0, "  ← la única fila\n     que no cambia",
        fontsize=10, fontweight="bold", color=AQUA, va="center", ha="left")

poner_titulo(ax, "Lo que define el OVR depende de la posición… salvo un atributo",
             "Correlación con el OVR calculada por separado dentro de cada posición",
             alto=(1.16, 1.06))
ax.text(0, -0.17,
        "Para un delantero manda el tiro (0,97); para un central, la defensa (0,98). "
        "Pero Reacciones se mantiene entre 0,89 y 0,92 en las once posiciones.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "10_correlacion_por_posicion.png")
plt.show()

**Este es el hallazgo más interesante del análisis.** Cada posición tiene su atributo
dominante — y tiene sentido futbolístico: un delantero vale por cómo define, un central
por cómo defiende. Pero **Reacciones es el único que se mantiene alto en las once
posiciones**. Es el denominador común de la valoración.

## Gráfico 11 — ¿Y los arqueros?

En el notebook 01 separamos a los arqueros porque sus columnas significan otra cosa.
Ahora podemos analizarlos por su cuenta y comprobar si la conclusión se sostiene.

In [ ]:
CARAS_GK = ["gk_diving", "gk_handling", "gk_kicking", "gk_reflexes", "gk_speed", "gk_positioning"]
EXTRA_GK = ["reactions", "composure", "jumping", "strength"]

corr_gk = arqueros[CARAS_GK + EXTRA_GK].corrwith(arqueros["ovr"]).sort_values(ascending=False)
colores_gk = [NARANJA if a in CARAS_GK else AZUL for a in corr_gk.index]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
posiciones = list(range(len(corr_gk)))[::-1]
ax.barh(posiciones, corr_gk.values, color=colores_gk, height=0.66)

for y, (nombre, valor) in zip(posiciones, corr_gk.items()):
    color = NARANJA if nombre in CARAS_GK else AZUL
    ax.text(valor + 0.015, y, coma(valor), ha="left", va="center",
            fontsize=10.5, fontweight="bold", color=color)

ax.set_yticks(posiciones, [NOMBRES.get(a, a) for a in corr_gk.index],
              fontsize=10.5, color=TINTA_SUAVE)
ax.set_xlim(0, 1.08)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "0,25", "0,50", "0,75", "1"])
ax.set_xlabel("Coeficiente de correlación con el OVR", labelpad=8)
ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

ax.legend(handles=[Patch(facecolor=NARANJA, label="Caras de la carta del arquero"),
                   Patch(facecolor=AZUL, label="Atributos de detalle")],
          loc="lower right", frameon=False, fontsize=10.5, labelcolor=TINTA_SUAVE,
          handlelength=1.1, handleheight=1.1)

poner_titulo(ax, "En los arqueros manda la colocación",
             "Las mismas cuentas sobre los 2.014 arqueros, con sus propias caras de carta",
             alto=(1.13, 1.04))
ax.text(0, -0.20,
        "Acá las caras visibles sí explican el OVR (0,88 a 0,97), al revés que en los jugadores de campo. "
        "Y Reacciones vuelve a aparecer alto (0,89):\nes el único atributo que manda en los dos datasets.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "11_correlacion_arqueros.png")
plt.show()

---
# PARTE 3 — Conclusiones

In [ ]:
conclusiones = pd.DataFrame([
    {"Hallazgo": "El atributo que más define el OVR",
     "Detalle": "Reacciones, con 0,896. No aparece en la cara de la carta."},
    {"Hallazgo": "Los cuatro primeros son invisibles",
     "Detalle": "Reacciones, Pase corto, Serenidad y Control del balón: ninguno se ve en la carta."},
    {"Hallazgo": "La cara más visible es la que menos importa",
     "Detalle": "PAC (ritmo) correlaciona 0,24, el número más grande de la carta y casi el más bajo de la tabla."},
    {"Hallazgo": "La respuesta cambia según la posición",
     "Detalle": "Tiro 0,97 en delanteros; Defensa 0,98 en centrales."},
    {"Hallazgo": "Salvo un atributo",
     "Detalle": "Reacciones se mantiene entre 0,89 y 0,92 en las once posiciones, y 0,89 en arqueros."},
    {"Hallazgo": "El físico casi no cuenta",
     "Detalle": "Altura -0,08 y Peso -0,05: prácticamente ninguna relación con el OVR."},
    {"Hallazgo": "El resultado es robusto",
     "Detalle": f"En el top 15, Pearson y Spearman difieren como máximo {coma(dif_top, 3)}."},
])
print(conclusiones.to_string(index=False))

## Respuesta a la pregunta del proyecto

> **¿Qué atributos de la carta determinan el OVR?**

El OVR lo definen sobre todo **atributos que el jugador no ve en la carta**. El que más
manda es **Reacciones (0,896)**, seguido de Pase corto, Serenidad y Control del balón.
Las seis caras visibles quedan por detrás, y la más prominente de todas — el ritmo —
es prácticamente irrelevante.

Dentro de cada posición la historia cambia: un delantero vale por su tiro y un central
por su defensa. Pero **Reacciones es el único atributo que se mantiene arriba en las
once posiciones y también en los arqueros**.

### Lo que este análisis NO prueba

Correlación no es causalidad. Que Reacciones acompañe al OVR no demuestra que EA use
ese atributo para calcularlo: podría ser que ambos dependan de una tercera cosa (la
calidad general del jugador). Además vimos que varios atributos están correlacionados
entre sí (PAS y DRI, 0,86), así que sus efectos no son independientes.

Para separar el efecto de cada atributo hace falta una **regresión múltiple**, que es
lo que corresponde a la etapa siguiente.